<a href="https://colab.research.google.com/github/npallab/genai_with_huggingface/blob/main/Basics_of_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction to LangChain Agents**

Welcome to this interactive exploration of LangChain Agents. While standard Large Language Models (LLMs) are incredibly capable of generating text, they are traditionally "frozen" in time—limited to the data they were trained on. Agents break these boundaries by using the LLM not just as a knowledge base, but as a reasoning engine.

In this notebook, we will move beyond static prompts and build a system that can reason, act, and solve complex problems by interacting with the real world. By equipping an LLM with "Tools"—such as web search, calculators, or custom Python functions—we enable it to decide which actions to take, execute those actions, and observe the results to reach a final goal. You aren't just writing a script; you are building a digital collaborator capable of autonomous decision-making.

**Chapter 1 :** Creating Basic Model without Tools using Google Generative AI

Pre Requisite : Google Gen AI API Key


In [4]:
#importing the libraries
import os # This will help in importing the API key
from google.colab import userdata # This library helps us in accessing the secrets stotred in colab
from langchain_google_genai import ChatGoogleGenerativeAI # This will help us create the model object

In [5]:
os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API')

In [6]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite") # Our model Object is here

In [7]:
from langchain.agents import create_agent # This library will help us in creating the agent

In [8]:
agent=create_agent(model,system_prompt="You are a helpful assistant expert in tech, reply in a few lines") # Agent is created , now we will invoke the agent

In [9]:
output=agent.invoke({"messages": [{"role": "user", "content": "What is Google Gemini"}]})

In [10]:
ai_output = output['messages'][-1].content #slicing to view only the AI Output

In [11]:
ai_output

"Google Gemini is a family of large language models (LLMs) developed by Google AI. It's designed to be multimodal, meaning it can understand and operate across different types of information, including text, code, audio, image, and video. Gemini comes in different sizes (Ultra, Pro, Nano) to suit various applications, from data centers to mobile devices."

In [12]:
type(output)

dict

Now we will create an agent with dynamic Model, this agent is capable of using two different models based on the input length

In [13]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse # we will use these to create the middleware that helps select models during the runtime

In [14]:
#Our models, these models with be selected based on the input
basic_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
advanced_model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [15]:
@wrap_model_call # decorator to make model selection
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

1. The Function Name
dynamic_model_selection: This is the label for the logic. Based on the name, this function likely looks at an incoming request and decides which AI model (e.g., Gemini Flash vs. Gemini Pro) is best suited to handle it.

2. The Parameters (The "Inputs")
Inside the parentheses (...), we define what the function needs to work:

request: ModelRequest:

request is the variable name.

: ModelRequest is a type hint. It means the request variable must be an object of the class ModelRequest (likely a custom class containing things like the user's prompt or budget).

handler:

This is a positional argument. Since it doesn't have a type hint here, it could technically be anything, but in this context, a "handler" is usually a function that gets called later to actually process the model logic.

3. The Return Type (The "Output")
-> ModelResponse:

The -> symbol points to the output.

This tells you that when this function finishes running, it will return an object of the type ModelResponse. This likely contains the AI's answer, metadata, and token usage.

In [16]:
#Creating the new Agent similar to the previous one
dynamic_agent = create_agent(
    model=basic_model,  # Default model
    middleware=[dynamic_model_selection]
)

In [17]:
#Calling the agent to generate the output
dynamic_output = dynamic_agent.invoke({
    "messages": [{"role": "user", "content": "what is Attention Mechanism in AI "}],
    "user_preferences": {"style": "technical", "verbosity": "crisp"},
})

In [18]:
dynamic_output.get('messages')[-1].content

'The **Attention Mechanism** in Artificial Intelligence is a technique that mimics human cognitive attention. It allows a model to **dynamically focus on the most relevant parts of the input data** when processing it and generating an output. Instead of treating all parts of the input equally, attention assigns different **weights** or **importance scores** to different parts, highlighting what\'s crucial for the current task.\n\nThink of it like this:\n\n*   **Reading a book:** When you read a sentence, your brain doesn\'t give equal importance to every single word. You focus on the keywords, the verbs, and the nouns that convey the core meaning. The rest of the words provide context.\n*   **Looking at a picture:** If you\'re asked to describe what\'s happening in a picture of a park, you\'ll likely focus on the people, the animals, and the prominent objects, rather than the blades of grass or the distant trees.\n\n**Why is Attention Important?**\n\nBefore attention mechanisms, many A

🛠️ **Understanding Tools in AI Agents**
In the development of AI Agents, Tools represent the "interfaces" or "capabilities" that allow an LLM to interact with the physical and digital world. Without tools, an AI is essentially a "brain in a jar"—it can reason and generate text, but it cannot act on its environment.

When you equip an agent with a tool, you provide it with a structured set of instructions on how to use a specific function, API, or database to solve problems that fall outside its static training data.

🧬 **The Anatomy of a Tool**
In the LangChain framework, a tool is more than just a function; it is a discrete package consisting of three vital components:

The Name: A unique identifier used by the system (e.g., Google Search).

The Description: The most critical part. This is the "instruction manual" the LLM reads to decide when and why it should invoke this tool.

The Function: The actual Python code or API call that executes once the agent triggers the tool.

Note: The LLM uses the tool's description as a prompt to understand its utility. A vague description often leads to "tool confusion" where the agent picks the wrong tool for the task.

All langchain Tools are available here : https://docs.langchain.com/oss/python/integrations/tools

In [18]:
# we will be using serper API to perform Google Search , Serper API can be obtained by signing up on https://serper.dev/
os.environ['SERPER_API_KEY']=userdata.get('SerperAPI')

In [19]:
from langchain_community.utilities import GoogleSerperAPIWrapper

search = GoogleSerperAPIWrapper()

In [21]:
#Testing Search
search.run("WHat is serper")

"Industry-leading SERP API, delivering lightning-fast Google search results in 1-2 seconds, at an unbeatable price starting at $0.30 per 1000 queries. This page covers how to use the Serper Google Search API within LangChain. Serper is a low-cost Google Search API that can be used to add answer box, knowledge ... The World's Fastest and Cheapest Google Search API. Experience unparalleled speed with our industry-leading SERP API, delivering lightning-fast Google search ... This tool is designed to perform a semantic search for a specified query from a text's content across the internet. https://estimate-ai.streamlit.app/ ▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭▭ In this video, I go over the SERPER API and how you ... Serper is a Google Search API specifically designed for developers looking to integrate Google's search capabilities into their projects. It ... Serper MCP Server is a Model Context Protocol (MCP) server that enables AI agents and LLMs to perform real-time Google Search queries—including .

In [23]:
from langchain.tools import tool
#Now we will create a tool
@tool
def intermediate_answer(query: str) -> str:
    """Useful for when you need to ask with search."""
    return search.run(query)

In [32]:
tools=[intermediate_answer]

In [33]:
#Creating a new agent with tool
dynamic_agent = create_agent(
    model=basic_model,  # Default model
    middleware=[dynamic_model_selection],
    tools=tools
)

In [34]:
tool_output=dynamic_agent.invoke({
    "messages": [{"role": "user", "content": "Whats the top news for today in India"}],
    "user_preferences": {"style": "technical", "verbosity": "crisp"},
})

In [35]:
clean_output=tool_output.get('messages')[-1].content

In [36]:
print(clean_output)

I found a few top news stories in India. Here are some highlights:

*   **Politics and Economy:** There's a discussion on SBI suggesting a "counter-intuitive" approach following the striking down of Trump's tariffs. Additionally, there are reports about the DMK intensifying its fight against communalism allegedly fostered by the BJP, and a former Assam Congress Chief joining the BJP due to alleged lack of funds for contesting polls.
*   **Security:** Security forces have reportedly killed two terrorists in Kishtwar. There are also reports about Delhi Police busting a Lashkar module.
*   **Sports:** In cricket, there's live coverage of the IND vs SA Super 8 match, with discussions on player selections and performances.
*   **Social Issues:** A tragic incident is reported where a mother and infant were burnt to death in an Indian state over witchcraft allegations. There are also reports of protests by the Youth Congress at an AI Summit, which a minister claims tarnished India's image.

P

**🔄 Dynamic Tool Selection:** Context-Aware Capability
In a production environment, giving an agent every available tool—known as Static Tooling—is often counterproductive. It creates "context bloat," where the model’s limited attention is wasted on irrelevant descriptions, leading to increased latency, higher token costs, and a higher probability of the model picking the wrong tool.

Dynamic Tool Selection addresses this by modifying the agent’s "toolbox" at runtime. This approach treats tools as adaptive assets rather than fixed code.

**Why Dynamic Selection Matters:**
Precision & Accuracy: By presenting only a handful of relevant tools (e.g., 5 instead of 50), you dramatically reduce "tool confusion" and hallucinations.

Security & Governance: You can implement Role-Based Access Control (RBAC). A "Manager" role might see a delete_user tool, while a "Guest" only sees view_profile.

Cost Optimization: Reducing the number of tool descriptions in your system prompt saves input tokens on every single turn of the conversation.

Stateful Adaptation: The agent can "unlock" tools as the conversation progresses. For example, a process_payment tool only appears once the agent has successfully helped the user build a shopping cart.

In [20]:
# we will first import the libraries :
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ToolCallRequest


In [50]:
class MultiDynamicToolMiddleware(AgentMiddleware):


    def wrap_model_call(self, request: ModelRequest, handler):
        # Inject ALL dynamic tools into the model's available toolset
        all_tools = [*request.tools, intermediate_answer]
        updated = request.override(tools=all_tools)
        return handler(updated)

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        # Handle execution of the dynamic tool
        if request.tool_call["name"] == "intermediate_answer":
            return handler(request.override(tool=intermediate_answer))


In [45]:
agent = create_agent(
    model=advanced_model,
    #tools=[intermediate_answer],  # Only static tools registered here
    middleware=[MultiDynamicToolMiddleware()],
)

In [46]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "New York's temperature today"}]
})

In [49]:
result['messages'][-1].content

[{'type': 'text',
  'text': 'As of today, Monday, February 23, 2026, the current temperature in New York City is approximately **29°F** (-2°C). \n\nThe weather today includes a **Blizzard Warning** from the National Weather Service, with snow and gusty winds expected. The forecast details for the day are:\n*   **High:** Around 43°F to 46°F (6°C to 8°C)\n*   **Low:** 31°F to 38°F (-1°C to 3°C)\n*   **Conditions:** Heavy snow at times with ice fog and high winds (NNE at 25 to 35 mph), leading to blizzard conditions. There is a high chance of precipitation (around 70–100%) with significant snow accumulation possible.\n\nPlease be advised that weather conditions are currently severe, and you should check local news for updates on travel and safety.',
  'extras': {'signature': 'EsQLCsELAb4+9vvaObGIktGJCEIFanwUxcylGfN/YNSAIo8Nt66ZUwFhi7XOtXM1+EKSLbFyQ4l46LcC8RhSjcsTtNOAY436Z3MJYYfu2de362Is9gmAUtYwn5m1Cyj4Sbw8F+yfF87shI4Q4Au35UMRIoRcyX3ha7AZH8kV44kOHnN3s2m5/y2UmK74qSnRoRK4CJIDaCHsMq/DQjx6kRxE